# Calculate Resource Outage Rates 

In [1]:
# Inputs
import pandas as pd
import numpy as np
import random
import os
import re
# Global cache to lock in the water state for each weather week
REGIONAL_WATER_CACHE = {}

## Thermal Unforced Outage Rates

In [2]:
df = pd.read_csv("Inputs/outage_scheduled_monthly.csv", header=0)

combined_cycle_scheduled = df[df['prime_mover'] == "combined_cycle"]
combustion_scheduled = df[df['prime_mover'] == "combustion_turbine"]
diesel_scheduled = combustion_scheduled
hydro_scheduled = df[df['prime_mover'] == "hydro_and_psh"]
nuclear_scheduled = df[df['prime_mover'] == "nuclear"]
steam_scheduled = df[df['prime_mover'] == "steam"]

In [3]:
print(hydro_scheduled)


      prime_mover  month  outage_frac
24  hydro_and_psh      1      0.11685
25  hydro_and_psh      2      0.12327
26  hydro_and_psh      3      0.13783
27  hydro_and_psh      4      0.13129
28  hydro_and_psh      5      0.11009
29  hydro_and_psh      6      0.08876
30  hydro_and_psh      7      0.08487
31  hydro_and_psh      8      0.09808
32  hydro_and_psh      9      0.13630
33  hydro_and_psh     10      0.18336
34  hydro_and_psh     11      0.16960
35  hydro_and_psh     12      0.12906


## Forced Outage Rates

### Thermal 

#### Baseline thermal forced outages

In [4]:
df = pd.read_csv("Inputs/outage_forced_temperature_murphy2019.csv", header=1)
df['deg_celsius'] = (df['deg_celsius'] * 9/5) + 32
df = df.rename(columns={'deg_celsius': "thi"})

# For summer 
combined_cycle_forced = df[df['prime_mover'] == "combined_cycle"]
combustion_forced = df[df['prime_mover'] == "combustion_turbine"]
diesel_forced = df[df['prime_mover'] == "diesel"]
hydro_forced = df[df['prime_mover'] == "hydro_and_psh"]
nuclear_forced = df[df['prime_mover'] == "nuclear"]
steam_forced = df[df['prime_mover'] == "steam"]

# For winter (if THI less than 20 F) 
cold_combined_cycle_forced = 0.25
cold_combustion_forced = 0.28
cold_diesel_forced = 0.0
cold_hydro_forced = 0.1
cold_nuclear_forced = 0.02
cold_steam_forced = 0.2


In [5]:
print(hydro_forced)

      prime_mover   thi  outage_frac
33  hydro_and_psh   5.0        0.070
34  hydro_and_psh  14.0        0.043
35  hydro_and_psh  23.0        0.031
36  hydro_and_psh  32.0        0.027
37  hydro_and_psh  41.0        0.026
38  hydro_and_psh  50.0        0.026
39  hydro_and_psh  59.0        0.027
40  hydro_and_psh  68.0        0.027
41  hydro_and_psh  77.0        0.024
42  hydro_and_psh  86.0        0.029
43  hydro_and_psh  95.0        0.082


#### Helper functions for thermal forced outages

In [6]:
# Helper function to collect nearest outage rate based on THI 
def get_nearest_outage(outage_data, target_thi):
    # Use pandas vectorized math for speed
    idx = (outage_data['thi'] - target_thi).abs().idxmin()
    return outage_data.loc[idx, 'outage_frac']
    

In [7]:
# Helper function for combustion turbines 
# Takes a 168-hour profile and applies an outage to each day based on the minimum THI and a random draw 
def apply_daily_rolling_outage(thi_list, scheduled_rate, normal_forced_rate, cold_forced_rate):

    daily_thi = np.array(thi_list).reshape(7, 24)
    weekly_availability = []

    for day_hours in daily_thi:
        # Check Stress: Use daily MIN for cold-weather forced outage logic
        min_thi = np.min(day_hours)
        
        # stress check
        if min_thi < 20:
            base_forced = cold_forced_rate
        else:
            # For normal bins, we use the average THI of the day to pick the bin
            base_forced = normal_forced_rate

        # Monte Carlo Logic: +/- 5% drawn once per day
        mc_shift = np.random.uniform(-0.1, 0.1)
        realized_forced = base_forced + mc_shift 
        
        # Calculation
        total_outage = scheduled_rate + realized_forced 
        on = 1.0 - total_outage 
        clamped_on = max(0.0, min(1.0, on)) 

        weekly_availability.extend([clamped_on] * 24)

    return weekly_availability

### Nonthermal Resources

In [8]:
long_solar = pd.read_csv("inputs/daily_solar_profiles_long.csv")
long_onshore = pd.read_csv("inputs/daily_onshore_profiles_long.csv")
long_offshore = pd.read_csv("inputs/daily_offshore_profiles_long.csv")

collapsed_solar = long_solar.groupby(['date', 'season', 'thi_extreme', 'thi_bin'], as_index=False)['solar_profile'].agg(list)
collapsed_wind_onshore = long_onshore.groupby(['date', 'season', 'thi_extreme', 'thi_bin'], as_index=False)['wind_profile'].agg(list)
collapsed_wind_offshore = long_offshore.groupby(['date', 'season', 'thi_extreme', 'thi_bin'], as_index=False)['wind_profile'].agg(list)\

# SOLAR 
# 1. Build the solar-specific dictionary
solar_bin_dfs = {}
for bin_val in collapsed_solar['thi_bin'].unique():
    solar_bin_dfs[bin_val] = collapsed_solar[collapsed_solar['thi_bin'] == bin_val].copy()

# 2. Extract bin ranges
solar_bin_ranges = {}
for bin_label in solar_bin_dfs.keys():
    clean_label = bin_label.replace('(', '').replace(']', '').replace(' ', '')
    lower, upper = map(float, clean_label.split(','))
    solar_bin_ranges[bin_label] = (lower, upper)

# 3. Find global min/max for solar
solar_sorted_labels = sorted(solar_bin_ranges.keys(), key=lambda x: solar_bin_ranges[x][0])
solar_lowest_bin = solar_sorted_labels[0]
solar_highest_bin = solar_sorted_labels[-1]

# 4. Pack into the "Solar Tuple"
# Order: (dictionary, ranges, lowest label, highest label)
solar_resource_tuple = (solar_bin_dfs, solar_bin_ranges, solar_lowest_bin, solar_highest_bin)

# ONSHORE WIND
# 1. Build the onshore-specific dictionary
onshore_bin_dfs = {}
for bin_val in collapsed_wind_onshore['thi_bin'].unique():
    onshore_bin_dfs[bin_val] = collapsed_wind_onshore[collapsed_wind_onshore['thi_bin'] == bin_val].copy()

# 2. Extract onshore bin ranges
onshore_bin_ranges = {}
for bin_label in onshore_bin_dfs.keys():
    clean_label = bin_label.replace('(', '').replace(']', '').replace(' ', '')
    lower, upper = map(float, clean_label.split(','))
    onshore_bin_ranges[bin_label] = (lower, upper)

# 3. Identify onshore bounds
onshore_sorted_labels = sorted(onshore_bin_ranges.keys(), key=lambda x: onshore_bin_ranges[x][0])
onshore_lowest_bin = onshore_sorted_labels[0]
onshore_highest_bin = onshore_sorted_labels[-1]

# 4. Pack into the "Onshore Tuple"
onshore_resource_tuple = (onshore_bin_dfs, onshore_bin_ranges, onshore_lowest_bin, onshore_highest_bin)

# OFFSHORE WIND 
# --- RUN THIS ONCE AT THE START FOR OFFSHORE ---

# 1. Build the offshore-specific dictionary
offshore_bin_dfs = {}
for bin_val in collapsed_wind_offshore['thi_bin'].unique():
    offshore_bin_dfs[bin_val] = collapsed_wind_offshore[collapsed_wind_offshore['thi_bin'] == bin_val].copy()

# 2. Extract offshore bin ranges
offshore_bin_ranges = {}
for bin_label in offshore_bin_dfs.keys():
    clean_label = bin_label.replace('(', '').replace(']', '').replace(' ', '')
    lower, upper = map(float, clean_label.split(','))
    offshore_bin_ranges[bin_label] = (lower, upper)

# 3. Identify offshore bounds
offshore_sorted_labels = sorted(offshore_bin_ranges.keys(), key=lambda x: offshore_bin_ranges[x][0])
offshore_lowest_bin = offshore_sorted_labels[0]
offshore_highest_bin = offshore_sorted_labels[-1]

# 4. Pack into the "Offshore Tuple"
offshore_resource_tuple = (offshore_bin_dfs, offshore_bin_ranges, offshore_lowest_bin, offshore_highest_bin)

In [9]:
def generate_solar_profile(thi_extreme): 
    # 1. THE UNPACKING LINE (Keep this)
    bin_dfs, bin_ranges, lowest_bin, highest_bin = solar_resource_tuple

    # 1. Handle the Absolute Extremes (Low/High)
    if thi_extreme <= bin_ranges[lowest_bin][0]:
        target_bin = lowest_bin
    elif thi_extreme > bin_ranges[highest_bin][1]:
        target_bin = highest_bin
    
    # 2. Handle the Middle (Including Gaps)
    else:
        # First, try a perfect hit (fast)
        for label, (low, high) in bin_ranges.items():
            if low < thi_extreme <= high:
                target_bin = label
                break
        
        # If no perfect hit (GAP FOUND), find the closest bin
        if target_bin is None:
            # Calculate distance to all bin boundaries
            # We check distance to both the 'low' and 'high' of every bin
            distances = {
                label: min(abs(thi_extreme - r[0]), abs(thi_extreme - r[1])) 
                for label, r in bin_ranges.items()
            }
            target_bin = min(distances, key=distances.get)

    # 3. PASTE THE SAFE SAMPLER HERE (Replacing your old loop)
    weekly_profile = []
    
    if target_bin not in bin_dfs:
        raise ValueError(f"Target bin {target_bin} not found! THI: {thi_extreme}")

    df_bin = bin_dfs[target_bin] 

    if df_bin.empty:
        # This is likely what is causing the NoneType error
        raise ValueError(f"Target bin {target_bin} is EMPTY in the data! THI: {thi_extreme}")

    for i in range(7):
        random_row = df_bin.sample(n=1).iloc[0]
        # Note: Change 'solar_profile' to 'wind_profile' for the wind functions!
        day_profile = random_row['solar_profile'] 
        
        if day_profile is None:
            print(f"Warning: Row in {target_bin} has None for profile.")
            
        weekly_profile.extend(day_profile)

    return weekly_profile

In [10]:
def generate_wind_profile(thi_extreme): 
    # 1. THE UNPACKING LINE (Keep this)
    bin_dfs, bin_ranges, lowest_bin, highest_bin = onshore_resource_tuple

    # 1. Handle the Absolute Extremes (Low/High)
    if thi_extreme <= bin_ranges[lowest_bin][0]:
        target_bin = lowest_bin
    elif thi_extreme > bin_ranges[highest_bin][1]:
        target_bin = highest_bin
    
    # 2. Handle the Middle (Including Gaps)
    else:
        # First, try a perfect hit (fast)
        for label, (low, high) in bin_ranges.items():
            if low < thi_extreme <= high:
                target_bin = label
                break
        
        # If no perfect hit (GAP FOUND), find the closest bin
        if target_bin is None:
            # Calculate distance to all bin boundaries
            # We check distance to both the 'low' and 'high' of every bin
            distances = {
                label: min(abs(thi_extreme - r[0]), abs(thi_extreme - r[1])) 
                for label, r in bin_ranges.items()
            }
            target_bin = min(distances, key=distances.get)

    # 3. PASTE THE SAFE SAMPLER HERE (Replacing your old loop)
    weekly_profile = []
    
    if target_bin not in bin_dfs:
        raise ValueError(f"Target bin {target_bin} not found! THI: {thi_extreme}")

    df_bin = bin_dfs[target_bin] 

    if df_bin.empty:
        # This is likely what is causing the NoneType error
        raise ValueError(f"Target bin {target_bin} is EMPTY in the data! THI: {thi_extreme}")

    for i in range(7):
        random_row = df_bin.sample(n=1).iloc[0]
        # Note: Change 'solar_profile' to 'wind_profile' for the wind functions!
        day_profile = random_row['wind_profile'] 
        
        if day_profile is None:
            print(f"Warning: Row in {target_bin} has None for profile.")
            
        weekly_profile.extend(day_profile)

    return weekly_profile

In [11]:
def generate_offshore_profile(thi_extreme): 
    # 1. THE UNPACKING LINE (Keep this)
    bin_dfs, bin_ranges, lowest_bin, highest_bin = offshore_resource_tuple

    # 1. Handle the Absolute Extremes (Low/High)
    if thi_extreme <= bin_ranges[lowest_bin][0]:
        target_bin = lowest_bin
    elif thi_extreme > bin_ranges[highest_bin][1]:
        target_bin = highest_bin
    
    # 2. Handle the Middle (Including Gaps)
    else:
        # First, try a perfect hit (fast)
        for label, (low, high) in bin_ranges.items():
            if low < thi_extreme <= high:
                target_bin = label
                break
        
        # If no perfect hit (GAP FOUND), find the closest bin
        if target_bin is None:
            # Calculate distance to all bin boundaries
            # We check distance to both the 'low' and 'high' of every bin
            distances = {
                label: min(abs(thi_extreme - r[0]), abs(thi_extreme - r[1])) 
                for label, r in bin_ranges.items()
            }
            target_bin = min(distances, key=distances.get)

    # 3. PASTE THE SAFE SAMPLER HERE (Replacing your old loop)
    weekly_profile = []
    
    if target_bin not in bin_dfs:
        raise ValueError(f"Target bin {target_bin} not found! THI: {thi_extreme}")

    df_bin = bin_dfs[target_bin] 

    if df_bin.empty:
        # This is likely what is causing the NoneType error
        raise ValueError(f"Target bin {target_bin} is EMPTY in the data! THI: {thi_extreme}")

    for i in range(7):
        random_row = df_bin.sample(n=1).iloc[0]
        # Note: Change 'solar_profile' to 'wind_profile' for the wind functions!
        day_profile = random_row['wind_profile'] 
        
        if day_profile is None:
            print(f"Warning: Row in {target_bin} has None for profile.")
            
        weekly_profile.extend(day_profile)

    return weekly_profile

### Final Helper Functions (combining forced and unforced)

In [12]:

# --- 1. TECHNOLOGY PERFORMANCE FUNCTIONS ---
# These currently return 1.0 (full availability) but take the THI list as input.

def calc_battery_storage(thi_list, month):
    return [1.0] * 168 # DONE 

def calc_biomass(thi_list, month):
    scheduled_rate = steam_scheduled.loc[steam_scheduled['month'] == month, 'outage_frac'].values[0]
    
    if month > 3 and month < 10: 
        thi = np.max(thi_list)
    else: 
        thi = np.min(thi_list)
    normal_forced_rate = get_nearest_outage(steam_forced, thi)
    
    cold_forced = cold_steam_forced
    
    return apply_daily_rolling_outage(thi_list, scheduled_rate, normal_forced_rate, cold_forced)        

def calc_ng_combined_cycle(thi_list, month):
    scheduled_rate = combined_cycle_scheduled.loc[combined_cycle_scheduled['month'] == month, 'outage_frac'].values[0]

    if month > 3 and month < 10: 
        thi = np.max(thi_list)
    else: 
        thi = np.min(thi_list)
    normal_forced_rate = get_nearest_outage(combined_cycle_forced, thi)
    
    cold_forced = cold_combined_cycle_forced
    
    return apply_daily_rolling_outage(thi_list, scheduled_rate, normal_forced_rate, cold_forced)        


def calc_ng_combustion_turbine(thi_list, month):
    scheduled_rate = combustion_scheduled.loc[combustion_scheduled['month'] == month, 'outage_frac'].values[0]

    if month > 3 and month < 10: 
        thi = np.max(thi_list)
    else: 
        thi = np.min(thi_list)
    normal_forced_rate = get_nearest_outage(combustion_forced, thi)
    
    cold_forced = cold_combustion_forced
    
    return apply_daily_rolling_outage(thi_list, scheduled_rate, normal_forced_rate, cold_forced)        

def calc_ng_steam(thi_list, month):
    scheduled_rate = steam_scheduled.loc[steam_scheduled['month'] == month, 'outage_frac'].values[0]

    if month > 3 and month < 10: 
        thi = np.max(thi_list)
    else: 
        thi = np.min(thi_list)
    normal_forced_rate = get_nearest_outage(steam_forced, thi)
    
    cold_forced = cold_steam_forced
    
    return apply_daily_rolling_outage(thi_list, scheduled_rate, normal_forced_rate, cold_forced)        


def calc_petroleum(thi_list, month):
    scheduled_rate = diesel_scheduled.loc[diesel_scheduled['month'] == month, 'outage_frac'].values[0]

    if month > 3 and month < 10: 
        thi = np.max(thi_list)
    else: 
        thi = np.min(thi_list)
    normal_forced_rate = get_nearest_outage(diesel_forced, thi)
    
    cold_forced = cold_diesel_forced
    
    return apply_daily_rolling_outage(thi_list, scheduled_rate, normal_forced_rate, cold_forced)        


# TODO 
def calc_solar_pv(thi_list, month):
    if month > 3 and month < 10: 
        season = "summer"
        thi_ext = np.max(thi_list)
    else: 
        season = "winter"
        thi_ext = np.min(thi_list)

    solar_prof = generate_solar_profile(thi_ext)
        
    return solar_prof

def calc_hydro(thi_list, month):
    # Plant-related outages 
    scheduled_rate = hydro_scheduled.loc[hydro_scheduled['month'] == month, 'outage_frac'].values[0]

    if month > 3 and month < 10: 
        thi = np.max(thi_list)
    else: 
        thi = np.min(thi_list)
    normal_forced_rate = get_nearest_outage(hydro_forced, thi)
    
    cold_forced = cold_hydro_forced

    '''
    # DOES THIS GO HERE?
    # Water availability outages 
    summer_water_availability = 0.25
    winter_water_availability = 0.4

    normal_forced_rate *= summer_water_availability
    cold_forced *= winter_water_availability
    '''
    return apply_daily_rolling_outage(thi_list, scheduled_rate, normal_forced_rate, cold_forced)        


def calc_coal(thi_list, month):
    scheduled_rate = steam_scheduled.loc[steam_scheduled['month'] == month, 'outage_frac'].values[0]

    if month > 3 and month < 10: 
        thi = np.max(thi_list)
    else: 
        thi = np.min(thi_list)
    normal_forced_rate = get_nearest_outage(steam_forced, thi)
    
    cold_forced = cold_steam_forced
    
    return apply_daily_rolling_outage(thi_list, scheduled_rate, normal_forced_rate, cold_forced)        


def calc_nuclear(thi_list, month):
    scheduled_rate = nuclear_scheduled.loc[nuclear_scheduled['month'] == month, 'outage_frac'].values[0]

    if month > 3 and month < 10: 
        thi = np.max(thi_list)
    else: 
        thi = np.min(thi_list)
    normal_forced_rate = get_nearest_outage(nuclear_forced, thi)
    
    cold_forced = cold_nuclear_forced
    
    return apply_daily_rolling_outage(thi_list, scheduled_rate, normal_forced_rate, cold_forced)        

def calc_wind_onshore(thi_list, month):
    # PJM Seasonality Logic
    if month in [4, 5, 6, 7, 8, 9]:
        season = "summer" # PJM Summer doldrums (low wind)
        thi_extreme = np.max(thi_list)
    elif month in [10, 11, 12, 1, 2, 3]:
        season = "winter" # Strong winter fronts
        thi_extreme = np.min(thi_list)

    return generate_wind_profile(thi_extreme)

def calc_wind_offshore(thi_list, month):
    # PJM Seasonality Logic
    if month in [4, 5, 6, 7, 8, 9]:
        season = "summer" # PJM Summer doldrums (low wind)
        thi_extreme = np.max(thi_list)
    elif month in [10, 11, 12, 1, 2, 3]:
        season = "winter" # Strong winter fronts
        thi_extreme = np.min(thi_list)
    
    return generate_offshore_profile(thi_extreme)

# TODO 
def calc_distributed_gen(thi_list, month):
    # PJM Seasonality
    if month > 3 and month < 10: 
        season = "summer"
        thi_ext = np.max(thi_list)
    else: 
        season = "winter"
        thi_ext = np.min(thi_list) 
        
    upv_profile = generate_solar_profile(thi_ext)

    tracking_scalar = 0.8 
    noise_scalar = np.random.uniform(0.97, 1.03)
    final_profile = [val * tracking_scalar * noise_scalar for val in upv_profile]
    return final_profile



def calc_pumped_hydro(thi_list, month):
    scheduled_rate = hydro_scheduled.loc[hydro_scheduled['month'] == month, 'outage_frac'].values[0]

    if month > 3 and month < 10: 
        thi = np.max(thi_list)
    else: 
        thi = np.min(thi_list)
    normal_forced_rate = get_nearest_outage(hydro_forced, thi)
    
    cold_forced = cold_hydro_forced
    
    return apply_daily_rolling_outage(thi_list, scheduled_rate, normal_forced_rate, cold_forced)        

# --- 2. COLUMN CATEGORIZATION MAPPING ---

def get_tech_function(col_name):
    name = col_name.lower()
    
    # Mapping logic based on your provided column headers
    if 'batteries' in name or 'battery' in name:
        return calc_battery_storage
    if 'solar' in name or 'pv' in name:
        return calc_solar_pv
    if 'offshore_wind' in name or 'offshorewind' in name:
        return calc_wind_offshore
    if 'wind' in name:  # Catch-all for onshore/landbased
        return calc_wind_onshore
    if 'nuclear' in name:
        return calc_nuclear
    if 'biomass' in name:
        return calc_biomass
    if 'hydroelectric' in name:
        if 'pumped' in name: return calc_pumped_hydro
        return calc_hydro
    if 'combined_cycle' in name or '1on1' in name:
        return calc_ng_combined_cycle
    if 'combustion_turbine' in name:
        return calc_ng_combustion_turbine
    if 'steam_turbine' in name:
        return calc_ng_steam
    if 'coal' in name:
        return calc_coal
    if 'petroleum' in name:
        return calc_petroleum
    if 'distributed' in name:
        return calc_distributed_gen
    
    return None # For indices like 'Time_Index'

# --- 3. MAIN PIECING LOGIC ---


### Generate Full Variability Data

In [14]:
# --- 1. & 2. TECHNOLOGY PERFORMANCE FUNCTIONS (Unchanged) ---
# (Keep your existing calc_solar, calc_wind, calc_hydro, etc. functions here)

# --- 3. CORE LOGIC ---

def generate_variability_data(template_path, thi_vector, month):
    """
    Generates the dictionary of data, but does NOT write to CSV yet.
    This separates the 'calculation' from the 'saving', which is cleaner for loops.
    """
    # Load the column list from the template
    df_template = pd.read_csv(template_path)
    final_cols = df_template.columns
    
    variability_data = {}
    variability_data['Time_Index'] = list(range(168))
    
    for col in final_cols:
        if col in ['Unnamed: 0', 'Time_Index']:
            continue
            
        func = get_tech_function(col)
        
        if func:
            # The stochastic magic happens here. 
            # Every time this is called, random.uniform gives new values.
            variability_data[col] = func(thi_vector, month)
        else:
            variability_data[col] = [1.0] * 168

        values_length = len(variability_data[col])
        if values_length != 168:
            raise ValueError(f"Generated data for column '{col}' has incorrect length: {values_length}")

    # Return DataFrame with correct column order
    df_output = pd.DataFrame(variability_data)

    # HYDRO WATER PATCH 
    # 1. Draw ONE regional water state for this entire realization
    if 3 < month < 10:  # Summer (Wet: 60%, Normal: 40%, Drought: 25%)
        water_state = np.random.choice([0.60, 0.40, 0.25], p=[0.20, 0.50, 0.30])
    else:               # Winter (Wet: 70%, Normal: 50%, Dry: 35%)
        water_state = np.random.choice([0.70, 0.50, 0.35], p=[0.30, 0.50, 0.20])
        
    # 2. Find all hydro columns in the dataframe
    # (Matches strings like 'IL1_conventional_hydroelectric_1')
    hydro_cols = [c for c in df_output.columns if 'conventional_hydroelectric' in c.lower()]
    
    # 3. Multiply all hydro columns by the single drawn state
    if hydro_cols:
        df_output[hydro_cols] = df_output[hydro_cols] * water_state
        

    return df_output.reindex(columns=final_cols)


# --- 4. EXECUTION LOOP ---

# Configuration
input_root = "Winter"         # Where your weather data lives
output_root = "Variability"   # Where the results go
template_file = "Inputs/Generators_variability.csv" 
month_for_simulation = 1     # January (Winter)

# Loop 1: The 10 Weather Cases (Summer_1 -> Summer_10)
for i in range(1, 11):
    case_name = f"{input_root}_{i}"
    file_name = f"{case_name}.csv"
    
    # 1. Setup Paths
    # Assuming input is inside a folder: Summer/Summer_1/weather.csv
    # Adjust 'weather.csv' to whatever your actual input filename is!
    input_path = os.path.join(input_root, file_name) 
    print(f"INPUT PATH: {input_path}")
    
    # Setup Output Directory: Variability/Summer_1/
    output_dir = os.path.join(output_root, case_name)
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"--- Processing Case: {case_name} ---")
    
    try:
        # 2. Load the specific THI vector for this week
        # Assuming the column is named 'THI', change if necessary
        df_weather = pd.read_csv(input_path)
        thi_vector = df_weather['thi'].tolist() 
    except FileNotFoundError:
        print(f"Error: Could not find weather file at {input_path}")
        continue

    # Loop 2: The 20 Stochastic Realizations (R1 -> R20)
    for r in range(1, 21):
        run_id = f"R{r}"
        
        # Generate the data (Recalculates wind/solar randomness)
        df_result = generate_variability_data(template_file, thi_vector, month_for_simulation)
        
        # Construct filename: Variability_Summer_1_R1.csv
        output_filename = f"Variability_{case_name}_{run_id}.csv"
        output_full_path = os.path.join(output_dir, output_filename)
        
        # Save
        df_result.to_csv(output_full_path, index=False)
        
        # Optional: Print less frequent status updates so console isn't flooded
        if r % 5 == 0:
            print(f"   Saved {output_filename}")

print("\nBatch generation complete.")

INPUT PATH: Winter\Winter_1.csv
--- Processing Case: Winter_1 ---
   Saved Variability_Winter_1_R5.csv
   Saved Variability_Winter_1_R10.csv
   Saved Variability_Winter_1_R15.csv
   Saved Variability_Winter_1_R20.csv
INPUT PATH: Winter\Winter_2.csv
--- Processing Case: Winter_2 ---
   Saved Variability_Winter_2_R5.csv
   Saved Variability_Winter_2_R10.csv
   Saved Variability_Winter_2_R15.csv
   Saved Variability_Winter_2_R20.csv
INPUT PATH: Winter\Winter_3.csv
--- Processing Case: Winter_3 ---
   Saved Variability_Winter_3_R5.csv
   Saved Variability_Winter_3_R10.csv
   Saved Variability_Winter_3_R15.csv
   Saved Variability_Winter_3_R20.csv
INPUT PATH: Winter\Winter_4.csv
--- Processing Case: Winter_4 ---
   Saved Variability_Winter_4_R5.csv
   Saved Variability_Winter_4_R10.csv
   Saved Variability_Winter_4_R15.csv
   Saved Variability_Winter_4_R20.csv
INPUT PATH: Winter\Winter_5.csv
--- Processing Case: Winter_5 ---
   Saved Variability_Winter_5_R5.csv
   Saved Variability_Winter_5